## Merging the deliveries.csv and matches.csv 

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px



In [3]:
matches= pd.read_csv(r'C:\first_data_science_proj\ipl-prediction\data\raw\matches.csv')
deliveries= pd.read_csv(r'C:\first_data_science_proj\ipl-prediction\data\raw\deliveries.csv')


In [4]:
matches.head()

,id,season,city,date,match_type,player_of_match,venue,team1,team2,toss_winner,toss_decision,winner,result,result_margin,target_runs,target_overs,super_over,method,umpire1,umpire2
0,335982,2007/08,Bangalore,2008-04-18,League,BB McCullum,M Chinnaswamy Stadium,Royal Challengers Bangalore,Kolkata Knight Riders,Royal Challengers Bangalore,field,Kolkata Knight Riders,runs,140.0,223.0,20.0,N,NaN,Asad Rauf,RE Koertzen
1,335983,2007/08,Chandigarh,2008-04-19,League,MEK Hussey,"Punjab Cricket Association Stadium, Mohali",Kings XI Punjab,Chennai Super Kings,Chennai Super Kings,bat,Chennai Super Kings,runs,33.0,241.0,20.0,N,NaN,MR Benson,SL Shastri
2,335984,2007/08,Delhi,2008-04-19,League,MF Maharoof,Feroz Shah Kotla,Delhi Daredevils,Rajasthan Royals,Rajasthan Royals,bat,Delhi Daredevils,wickets,9.0,130.0,20.0,N,NaN,Aleem Dar,GA Pratapkumar
3,335985,2007/08,Mumbai,2008-04-20,League,MV Boucher,Wankhede Stadium,Mumbai Indians,Royal Challengers Bangalore,Mumbai Indians,bat,Royal Challengers Bangalore,wickets,5.0,166.0,20.0,N,NaN,SJ Davis,DJ Harper
4,335986,2007/08,Kolkata,2008-04-20,League,DJ Hussey,Eden Gardens,Kolkata Knight Riders,Deccan Chargers,Deccan Chargers,bat,Kolkata Knight Riders,wickets,5.0,111.0,20.0,N,NaN,BF Bowden,K Hariharan


In [5]:
deliveries.head()

,match_id,inning,batting_team,bowling_team,over,ball,batter,bowler,non_striker,batsman_runs,extra_runs,total_runs,extras_type,is_wicket,player_dismissed,dismissal_kind,fielder
0,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,1,SC Ganguly,P Kumar,BB McCullum,0,1,1,legbyes,0,NaN,NaN,NaN
1,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,2,BB McCullum,P Kumar,SC Ganguly,0,0,0,NaN,0,NaN,NaN,NaN
2,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,3,BB McCullum,P Kumar,SC Ganguly,0,1,1,wides,0,NaN,NaN,NaN
3,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,4,BB McCullum,P Kumar,SC Ganguly,0,0,0,NaN,0,NaN,NaN,NaN
4,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,5,BB McCullum,P Kumar,SC Ganguly,0,0,0,NaN,0,NaN,NaN,NaN


In [6]:
print(matches.shape)
print(deliveries.shape)


print(f'\n column in matches: {matches.columns.tolist()}')
print(f'\n column in deliveries: {deliveries.columns.tolist()}')

(1095, 20)
(260920, 17)

 column in matches: ['id', 'season', 'city', 'date', 'match_type', 'player_of_match', 'venue', 'team1', 'team2', 'toss_winner', 'toss_decision', 'winner', 'result', 'result_margin', 'target_runs', 'target_overs', 'super_over', 'method', 'umpire1', 'umpire2']

 column in deliveries: ['match_id', 'inning', 'batting_team', 'bowling_team', 'over', 'ball', 'batter', 'bowler', 'non_striker', 'batsman_runs', 'extra_runs', 'total_runs', 'extras_type', 'is_wicket', 'player_dismissed', 'dismissal_kind', 'fielder']


In [7]:
#  column audit before the cleaning 

print('--'*50)
print('matches.csv -> key columns audit')
print('='*50)

print('\n [result] unique values')
print(matches['result'].value_counts(dropna=False).to_dict())
print(' keep only : runs,wickets')

print('\n [method] unique values (NaN = no DL):')
print(matches['method'].value_counts(dropna=False).to_dict())
print(' drop rows where method is not nan (dl applied)')

print('\n [super_over] unique values')
print(matches['super_over'].value_counts().to_dict())
print(' keep only : N')

print('\n[winner ] null count:', matches['winner'].isna().sum())
print(' inner join will eliminate these anyway')

print('--'*50)
print('deliveries.csv -> key columns audit')
print('='*50)

print('\n [inning] unique values:', sorted(deliveries['inning'].unique()))
print('keep only 1,2 (drop 3,4 =super overs)')

print('\n [is_wicket] values : ', sorted(deliveries['is_wicket'].unique()))
print('already 0/1 - no derivation needed')

print('\n [over] range:', deliveries['over'].min(),'to',deliveries['over'].max())
print('0-indexed! Need to +1 to make it 1-indexed (1 to 20)')

print('\n [extras_type] top values')
print(deliveries['extras_type'].value_counts(dropna=False).head(6).to_dict())


----------------------------------------------------------------------------------------------------
matches.csv -> key columns audit

 [result] unique values
{'wickets': 578, 'runs': 498, 'tie': 14, 'no result': 5}
 keep only : runs,wickets

 [method] unique values (NaN = no DL):
{nan: 1074, 'D/L': 21}
 drop rows where method is not nan (dl applied)

 [super_over] unique values
{'N': 1081, 'Y': 14}
 keep only : N

[winner ] null count: 5
 inner join will eliminate these anyway
----------------------------------------------------------------------------------------------------
deliveries.csv -> key columns audit

 [inning] unique values: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6)]
keep only 1,2 (drop 3,4 =super overs)

 [is_wicket] values :  [np.int64(0), np.int64(1)]
already 0/1 - no derivation needed

 [over] range: 0 to 19
0-indexed! Need to +1 to make it 1-indexed (1 to 20)

 [extras_type] top values
{nan: 246795, 'wides': 8380, 'legbyes': 4001, '

In [8]:
# team name standarization

team_name_mapping={
    
    'Delhi Daredevils'             : 'DC',
    'Delhi Capitals'               : 'DC',
    
    'Deccan Chargers'              : 'SRH',
    'Sunrisers Hyderabad'          : 'SRH',
  
    'Kings XI Punjab'              : 'PBKS',
    'Punjab Kings'                 : 'PBKS',
 
    'Royal Challengers Bangalore'  : 'RCB',
    'Royal Challengers Bengaluru'  : 'RCB',

    'Rising Pune Supergiant'       : 'RPS',
    'Rising Pune Supergiants'      : 'RPS',
  
    'Gujarat Lions'                : 'GL',
    'Gujarat Titans'               : 'GT',
    'Lucknow Super Giants'         : 'LSG',
    'Mumbai Indians'               : 'MI',
    'Chennai Super Kings'          : 'CSK',
    'Kolkata Knight Riders'        : 'KKR',
    'Rajasthan Royals'             : 'RR',
    'Kochi Tuskers Kerala'         : 'KTK',
    'Pune Warriors'                : 'PWI',
    'Pune Warriors India'          : 'PWI',
}

print(f'team name mapping : covers {len(team_name_mapping)} name variants')
print('key mapping')
seen=set()
for k,v in team_name_mapping.items():
    if v not in seen:
        variants=[k2 for k2, v2 in team_name_mapping.items() if v2==v]
        if len(variants)>1:
            print(f' {v:6s}<- {variants}')

        seen.add(v)

team name mapping : covers 20 name variants
key mapping
 DC    <- ['Delhi Daredevils', 'Delhi Capitals']
 SRH   <- ['Deccan Chargers', 'Sunrisers Hyderabad']
 PBKS  <- ['Kings XI Punjab', 'Punjab Kings']
 RCB   <- ['Royal Challengers Bangalore', 'Royal Challengers Bengaluru']
 RPS   <- ['Rising Pune Supergiant', 'Rising Pune Supergiants']
 PWI   <- ['Pune Warriors', 'Pune Warriors India']


In [9]:
#  cleaning in matches.csv

print(f'shape of the matches dataset before cleaning: {matches.shape}')

matches['date']=pd.to_datetime(matches['date'])
print(matches['date'].dtype)

before=len(matches)
matches=matches[matches['method'].isna()].copy()
print(f' {before}-> {len(matches)} (rmoved {before-len(matches)})')

before=len(matches)
matches=matches[matches['result'].isin(['runs','wickets'])].copy()
print(f' {before}-> {len(matches)} (removed {before-len(matches)})')

before=len(matches)
matches=matches[matches['super_over']=='N'].copy()
print(f' super over : {before}-> {len(matches)} (removed{before-len(matches)})')


for col in ['team1','team2','toss_winner','winner']:
    matches[col]=matches[col].replace(team_name_mapping)

unique_teams=pd.concat([matches.team1,matches.team2]).nunique()
print(f' team names : {unique_teams} unique teams after standarizations')


drop_cols=[c for c in [
    'umpire1','umpire2','city','player_of_match','method','super_over','match_type','target_overs'
] if c in matches.columns]

matches.drop(columns=drop_cols,inplace=True)

print(f' dropped columns: {drop_cols}')

print(f'\n final matches shape: {matches.shape}')

print(f' columns kept: {matches.columns.tolist()}')
print(f'remaining nulls : {matches.isnull().sum().sum()}')


shape of the matches dataset before cleaning: (1095, 20)
datetime64[us]
 1095-> 1074 (rmoved 21)
 1074-> 1055 (removed 19)
 super over : 1055-> 1055 (removed0)
 team names : 14 unique teams after standarizations
 dropped columns: ['umpire1', 'umpire2', 'city', 'player_of_match', 'method', 'super_over', 'match_type', 'target_overs']

 final matches shape: (1055, 12)
 columns kept: ['id', 'season', 'date', 'venue', 'team1', 'team2', 'toss_winner', 'toss_decision', 'winner', 'result', 'result_margin', 'target_runs']
remaining nulls : 0


In [10]:
# clean the deliveries dataset

print('cleaning deliveries csv')
print(f'starting shape: {deliveries.shape}')
print()

deliveries['over']=deliveries['over']+1
print(f' over range : {deliveries.over.min()} to {deliveries.over.max()}')

for col in ['batting_team','bowling_team']:
    deliveries[col]=deliveries[col].replace(team_name_mapping)
print(f' team names standarized in deliveries')

before=len(deliveries)
deliveries=deliveries[deliveries['inning'].isin([1,2])].copy()
print(f' super over balls : {before}-> {len(deliveries)} (removed {before-len(deliveries)})')

valid_ids=set(matches['id'].unique())
before=len(deliveries)
deliveries=deliveries[deliveries['match_id'].isin(valid_ids)].copy()
print(f' invalid ids : {before}-> {len(deliveries)} (removed {before-len(deliveries)})')

print(f'\n is_wicket values : {sorted(deliveries["is_wicket"].unique())}')
print(f'total wickets : {deliveries["is_wicket"].sum():,}')
print(f'batter col : {"batter" in deliveries.columns}')
print(f'\n final deliveries shape: {deliveries.shape}')


cleaning deliveries csv
starting shape: (260920, 17)

 over range : 1 to 20
 team names standarized in deliveries
 super over balls : 260920-> 260759 (removed 161)
 invalid ids : 260759-> 253150 (removed 7609)

 is_wicket values : [np.int64(0), np.int64(1)]
total wickets : 12,534
batter col : True

 final deliveries shape: (253150, 17)


In [11]:
# pre - merge validations

ids_in_del=set(deliveries['match_id'].unique())
ids_in_mat=set(matches['id'].unique())

only_in_del=ids_in_del - ids_in_mat
only_in_mat=ids_in_mat - ids_in_del
common= ids_in_del & ids_in_mat

print('--'* 55)
print('pre-merge validation')
print('='* 55)
print(f' match_ids only in deliveries : {len(only_in_del)}')
print(f' match_ids only in matches : {len(only_in_mat)}')
print(f'common match ids : {len(common)}')
print(f'matches rows : {len(matches)}')
print(f'ids match rows : {len(common)==len(matches)}')

if len(only_in_del)==0 and len(only_in_mat)==0:
    print('\n validation passed : safe to merge')
else:
    print('\n mismatch - investigate before merging')

    if only_in_del:
        print(f' ids in deliveries not in matches : {list(only_in_del)[:5]}')
    if only_in_mat:
        print(f' ids in matches not in deliveries : {list(only_in_mat)[:5]}')
        

--------------------------------------------------------------------------------------------------------------
pre-merge validation
 match_ids only in deliveries : 0
 match_ids only in matches : 0
common match ids : 1055
matches rows : 1055
ids match rows : True

 validation passed : safe to merge


In [12]:
# merging the csv's (inner join)

match_cols=[
    'id','season',
    'date','venue','team1','team2','toss_winner','toss_decision','winner','result','result_margin','target_runs',
]

merged=deliveries.merge(matches[match_cols],left_on='match_id',right_on='id',how='inner')

merged.drop(columns=['id'],inplace=True)


print(f'  deliveries (clean) : {deliveries.shape}')
print(f'  matches (clean)    : {matches[match_cols].shape}')
print(f'  merged result      : {merged.shape}')
print(f'\n  Expected: ~same row count as clean deliveries')
print(f'  Row difference    : {abs(len(deliveries) - len(merged))} rows')
print(f'\nMerged columns ({len(merged.columns)}):')
print(merged.columns.tolist())



  deliveries (clean) : (253150, 17)
  matches (clean)    : (1055, 12)
  merged result      : (253150, 28)

  Expected: ~same row count as clean deliveries
  Row difference    : 0 rows

Merged columns (28):
['match_id', 'inning', 'batting_team', 'bowling_team', 'over', 'ball', 'batter', 'bowler', 'non_striker', 'batsman_runs', 'extra_runs', 'total_runs', 'extras_type', 'is_wicket', 'player_dismissed', 'dismissal_kind', 'fielder', 'season', 'date', 'venue', 'team1', 'team2', 'toss_winner', 'toss_decision', 'winner', 'result', 'result_margin', 'target_runs']


In [13]:
key_cols=['match_id','winner','season','venue','batting_team']
nulls=merged[key_cols].isnull().sum()

print(f'\n[1] null in key columns')
print(nulls.to_string())
assert nulls.sum()==0, f' fail: {nulls[nulls > 0].to_dict()}'
print('pass')

print(f'\n[2] over range : {merged.over.min()} to {merged.over.max()}')
assert merged.over.min() ==1
assert merged.over.max()==20
print('pass')

inning_vals=sorted(merged['inning'].unique())
print(f'\n [3] inning present: {inning_vals} ')
assert inning_vals==[1,2],f' fail: unexpected inning {inning_vals}'
print('pass')

wkt_vals=set(merged['is_wicket'].unique())
print(f'\n [4] is_wicket values : {sorted(wkt_vals)} (expected : {{0,1}})')
assert wkt_vals=={0,1}, f' fail:unexpected-values {wkt_vals}'
print('pass')

merged_match_count=merged['match_id'].nunique()
clean_match_count=len(matches)
print(f'\n [5] unique matches in merged : {merged_match_count}')
print(f' clean matches rows : {clean_match_count}')
assert merged_match_count==clean_match_count,f' fail: {merged_match_count}=! {clean_match_count}'

print('pass')

print('\n '+'='*55)
print('all 5 check passed')




[1] null in key columns
match_id        0
winner          0
season          0
venue           0
batting_team    0
pass

[2] over range : 1 to 20
pass

 [3] inning present: [np.int64(1), np.int64(2)] 
pass

 [4] is_wicket values : [np.int64(0), np.int64(1)] (expected : {0,1})
pass

 [5] unique matches in merged : 1055
 clean matches rows : 1055
pass

all 5 check passed


In [14]:
# create the y target lable
merged['batting_team_wins']=(merged['batting_team']==merged['winner']).astype(int)

print('y label: batting_team_wins')
print('distribution (full merged-all innings):')
dist=merged['batting_team_wins'].value_counts(normalize=True).round(3)
print(f' 0 (batting team loses): {dist[0]:.1%}')
print(f' 1 (batting team wins) : {dist[1]:.1%}')

second_inn=merged[merged['inning']==2].copy()

print(f'\n 2nd innings rows : {len(second_inn):,}')
print(f'2nd innings y dist :')
dist2=second_inn['batting_team_wins'].value_counts(normalize=True).round(3)
print(f'  0 (chasing team loses): {dist2[0]:.1%}')
print(f'  1 (chasing team wins) : {dist2[1]:.1%}')
print('  → should be close to 50/50 — roughly half the time the chase succeeds')

y label: batting_team_wins
distribution (full merged-all innings):
 0 (batting team loses): 51.0%
 1 (batting team wins) : 49.0%

 2nd innings rows : 122,622
2nd innings y dist :
  0 (chasing team loses): 48.0%
  1 (chasing team wins) : 52.0%
  → should be close to 50/50 — roughly half the time the chase succeeds


In [15]:
print(f'Shape : {merged.shape}')
print(f'\nDtypes:')
print(merged.dtypes.to_string())
print(f'\nNull counts:')
null_counts = merged.isnull().sum()
null_nonzero = null_counts[null_counts > 0]
if len(null_nonzero) == 0:
    print('  Zero nulls in all key columns ✓')
else:
    print(null_nonzero.to_string())
    print('  ↑ These are okay: player_dismissed/fielder null = no wicket on that ball')

print(f'\nSample row:')
merged.head(2)

Shape : (253150, 29)

Dtypes:
match_id                      int64
inning                        int64
batting_team                    str
bowling_team                    str
over                          int64
ball                          int64
batter                          str
bowler                          str
non_striker                     str
batsman_runs                  int64
extra_runs                    int64
total_runs                    int64
extras_type                     str
is_wicket                     int64
player_dismissed                str
dismissal_kind                  str
fielder                         str
season                          str
date                 datetime64[us]
venue                           str
team1                           str
team2                           str
toss_winner                     str
toss_decision                   str
winner                          str
result                          str
result_margin               float6

,match_id,inning,batting_team,bowling_team,over,ball,batter,bowler,non_striker,batsman_runs,...,venue,team1,team2,toss_winner,toss_decision,winner,result,result_margin,target_runs,batting_team_wins
0,335982,1,KKR,RCB,1,1,SC Ganguly,P Kumar,BB McCullum,0,...,M Chinnaswamy Stadium,RCB,KKR,RCB,field,KKR,runs,140.0,223.0,1
1,335982,1,KKR,RCB,1,2,BB McCullum,P Kumar,SC Ganguly,0,...,M Chinnaswamy Stadium,RCB,KKR,RCB,field,KKR,runs,140.0,223.0,1


In [17]:
# save the three csv files
matches.to_csv(r'C:\first_data_science_proj\ipl-prediction\data\processed\matches_clean.csv',index=False)
merged.to_csv(r'C:\first_data_science_proj\ipl-prediction\data\processed\merged_clean.csv',index=False)
second_inn.to_csv(r'C:\first_data_science_proj\ipl-prediction\data\processed\second_innings_clean.csv',index=False)

print(f' matches_clean.csv {matches.shape}')
print(f' merged_clean.csv:{merged.shape}')
print(f'second_innings_clean.csv:{second_inn.shape}')


 matches_clean.csv (1055, 12)
 merged_clean.csv:(253150, 29)
second_innings_clean.csv:(122622, 29)


In [18]:
test = pd.read_csv(r'C:\first_data_science_proj\ipl-prediction\data\processed\merged_clean.csv')
assert 'Unnamed: 0' not in test.columns, 'index=False failed'
assert test.isnull().sum()[['match_id', 'winner', 'batting_team']].sum() == 0
assert test['batting_team_wins'].isin([0, 1]).all()
print(f'  merged_clean.csv reloaded  : {test.shape}  ✓')
print('  No unnamed:0 column        : ✓')
print('  Key columns null-free      : ✓')
print('  y label is 0/1 only        : ✓')
print()
print('=' * 55)
print('NOTEBOOK 03 COMPLETE')
print('Next step → run 04_type1_fe.ipynb')
print('=' * 55)

C:\Users\user\AppData\Local\Temp\ipykernel_37648\1896867021.py:1: DtypeWarning: Columns (0: season) have mixed types. Specify dtype option on import or set low_memory=False.
  test = pd.read_csv(r'C:\first_data_science_proj\ipl-prediction\data\processed\merged_clean.csv')


  merged_clean.csv reloaded  : (253150, 29)  ✓
  No unnamed:0 column        : ✓
  Key columns null-free      : ✓
  y label is 0/1 only        : ✓

NOTEBOOK 03 COMPLETE
Next step → run 04_type1_fe.ipynb
